# 01 — The frame index, and deciding a frame's type from its pixels

Build step 1 and 1.5 (`DECISIONS` D16). Two questions:

1. Can we read a frame and split it into CFA sub-planes without ever debayering (D4)?
2. Can we decide what a frame *is* from its own pixels, given that neither the folder
   nor `IMAGETYP` is trustworthy (D18)?

The output is `results/frame_index.csv` — one row per frame in the archive, carrying the
trusted capture settings, a handful of measured features, the measured type, the declared
type, and whether they agree. Three consumers are already waiting on it: dark/flat matching
(D9), per-frame sky extraction (the model's sky term), and phase-2 scheduling.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropix import cfa, fits as F

pd.set_option("display.width", 200)
INDEX = pathlib.Path("..") / "results" / "frame_index.csv"


def load_index():
    """Read the index, with the few derived columns every cell below wants.

    Safe to run while a refresh is in progress.  `refresh_index` writes a temp
    file and `os.replace`s it, which is atomic, so a reader sees either the
    previous complete file or the new one -- never a half-written one.  And
    rows are never removed (D19), so a partial index is simply a shorter one.
    Rows arrive in path order, which means bias, then dark, then flat, then
    light: a run still in progress has no lights yet.
    """
    idx = pd.read_csv(INDEX)
    idx["root"] = idx.path.str.extract(r"_by_type\\([^\\]+)", expand=False)
    idx["night"] = idx.date_obs.str[:10]
    # recompute rather than trusting the stored column, so the notebook is not
    # hostage to how csv round-tripped True/False/blank
    idx["agrees"] = idx.declared_type.str.lower() == idx.measured_type
    ok = idx[idx.status == "ok"].copy()
    print(f"{len(idx)} rows, {len(ok)} readable, "
          f"indexed {idx.indexed_at.min()} .. {idx.indexed_at.max()}")
    print("by folder:", dict(ok.root.value_counts()))
    return idx, ok

def reclassify(ok):
    """Re-derive measured_type from the stored feature columns.

    Every input `classify` takes is a column in the index, so a change to the
    classifier can be replayed over the archive without re-reading a single
    frame.  That is worth knowing: it is what makes the thresholds revisable.
    """
    return ok.apply(lambda r: F.classify(r, r.exptime), axis=1)


## Why classification reads row-blocks, not frames

The archive is ~15,000 frames of 16.6 MB on a network drive. Timed on this rig:

| operation | cost |
|---|---|
| open + read header | 0.09 s |
| open + read 32 rows via `hdu.section` | 0.16 s |
| open + read all 16.6 MB | 2.7 s |

Reading whole frames is an 11-hour pass; sampling is well under one. Nothing is lost,
because classification wants *statistics* and statistics converge long before you have
read four million pixels.

The blocks are **contiguous** rather than strided, and that is not an accident. Two of the
features are spatial — whether a bright pixel has a bright neighbour — and row striding
destroys exactly that. They are **spread down the frame** so that vignetting and amp glow,
both corner-weighted, are sampled rather than missed.

In [ ]:
idx, ok = load_index()

# a light if the index has reached them, otherwise whatever it has -- the point
# of this cell is the read geometry, not the frame
lights = ok[ok.measured_type == "light"]
path = lights.path.iloc[0] if len(lights) else ok.path.iloc[0]
if not len(lights):
    print("no lights indexed yet; showing", ok.measured_type.iloc[0], "instead\n")

blocks, header = F.sample_blocks(path)
print(path)
print(f"{len(blocks)} blocks of {blocks[0].shape}, dtype {blocks[0].dtype}")
print(f"{sum(b.nbytes for b in blocks) / 1e6:.2f} MB read, "
      f"of {header['NAXIS1'] * header['NAXIS2'] * 2 / 1e6:.1f} MB in the frame")

planes = cfa.split(blocks[0])
{name: (p.shape, float(np.median(p))) for name, p in planes.items()}

## The 12-bit ADC, seen directly

`FINDINGS` records this as a *suspicion*: the ASI585's ADC is 12-bit, ASIAIR stores 16-bit
FITS, and the values are believed to be bit-shifted x16. If that is true then every stored
value is an exact multiple of 16 — the low four bits are padding, not measurement.

That is a testable claim about the pixels, so we test it rather than believe it.

In [ ]:
sample = np.concatenate([b.ravel() for b in blocks])
frac16 = float(np.mean(sample % 16 == 0))
print(f"fraction of values that are exact multiples of 16: {frac16:.6f}")
print(f"distinct low-4-bit values present: {sorted(set((sample % 16).tolist()))}")
print(f"max value seen: {sample.max()}  =  {sample.max() // 16} x 16")

**Consequence.** One count in the file is not one count of the ADC — it is 1/16 of one, and
it can never be occupied. Any e-/ADU figure must state which ADU it means. The header's
`EGAIN` is in *12-bit* ADU, so using it against 16-bit file values inflates electron counts
16x. This is the unit trap that gates build step 3.

A second consequence is less obvious and more dangerous: the quantiser is coarse relative to
the read noise at low gain. If sigma is comparable to one step, a robust spread estimated
from a single frame measures the ADC rather than the sensor. We check that below on the
index, where every gain is represented.

## The classifier

Four decisions, in an order chosen so the cheapest and most trustworthy evidence goes first.

1. **Bias** — the shortest exposure the camera takes. Exposure is a *capture setting*, which
   D18 leaves trusted, so this is settled without a pixel argument.
2. **Clipped** — saturated across the frame. This is the one branch that is an *inference*
   rather than a measurement, and it is marked as such: every feature is degenerate (level
   pinned to full scale, sigma and clump zero), so there is nothing left to measure. The
   fallback is exposure — every flat in this archive is 1–3 s, so a clipped long exposure is
   a light that ran into dawn, not a flat. Saturation itself stays recorded in `sat_frac`,
   a *quality* attribute orthogonal to what the frame is (D25).
3. **Flat** — level an order of magnitude above the pedestal. Flats sit at tens of thousands
   of ADU; darks and lights sit within a few hundred of the pedestal. No overlap.
4. **Dark vs light** — the hard case, since at equal exposure they share a pedestal and a
   noise floor. What separates them is *shape*, not level: a star is a PSF spread over
   several pixels, a hot pixel is one defective site.

The dark/light feature is therefore connectedness. Among pixels above `median + 5 sigma`,
what fraction have an above-threshold neighbour? Measured **inside a sub-plane**, where
neighbouring samples share a colour filter and the same optical scale — on the mosaic they
do not, and the comparison would be meaningless.

Horizontal and vertical connectedness are counted separately and the **weaker** one is used.
A hot *column* — an ordinary CMOS defect — is vertically connected and would otherwise read
as a star. A real PSF is connected both ways.

In [ ]:
idx, ok = load_index()

FEATURES = ["level", "sigma", "tail_frac", "clump_frac", "clump_h", "clump_v"]
ok.groupby("measured_type")[FEATURES].median()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for t, g in ok.groupby("measured_type"):
    ax.scatter(g.level.clip(1, None), g.clump_frac, s=6, alpha=.4, label=t)
ax.set_xscale("log")
ax.axvline(F.FLAT_MIN_LEVEL * F.FULL_SCALE, ls="--", c="k", lw=.8)
ax.axhline(F.LIGHT_MIN_CLUMP, ls="--", c="k", lw=.8)
ax.set_xlabel("level  (median across CFA planes, file ADU)")
ax.set_ylabel("clump fraction  (weaker axis)")
ax.set_title("the two cuts that do the work")
ax.legend()
plt.show()

## Does the label agree with the pixels?

D18 predicted disagreement: some flats and darks were captured under a Light subframe type.
The index quantifies it. Disagreements are a **finding and a cleanup work-list** (D20), not
a hazard — nothing downstream reads `IMAGETYP`.

In [ ]:
print(pd.crosstab(ok.declared_type, ok.measured_type, margins=True))
print()
print(pd.crosstab(ok.root, ok.measured_type, margins=True))

In [ ]:
# only frames that carry a label can disagree with one
labelled = ok[ok.declared_type.notna()]
disagree = labelled[~labelled.agrees]
print(f"{len(disagree)} of {len(labelled)} labelled frames disagree with their pixels")
disagree.groupby(["declared_type", "measured_type"]).size()

## Quantisation vs read noise, across the gain axis

The claim to test: at low gain the single-frame robust sigma collapses onto multiples of
1.4826 x 16 = 23.72, because MAD lands on exactly one ADC step. Where that happens, the
number is a property of the quantiser and says nothing about the sensor.

In [ ]:
cal = ok[ok.measured_type.isin(["bias", "dark"])]
print(cal.groupby("gain").sigma.describe()[["count", "min", "50%", "max"]])
print()
print("quantisation step in sigma units: 1.4826 * 16 =", 1.4826 * 16)

## The NGC7000 ladder (D7)

The primary validation dataset. Six sub-exposure rungs, each totalling exactly 9600 s,
interleaved across four nights. Before trusting that description, confirm it from the index:
the counts, the total integration per rung, the gain, and the achieved temperature.

In [ ]:
lad = ok[ok.path.str.contains("NGC7000_tests", regex=False)]
print("frames:", len(lad))
print(pd.crosstab([lad.gain, lad.exptime], lad.measured_type, margins=True))
print()
print(lad.groupby(["gain", "exptime"]).agg(n=("exptime", "size"), total_s=("exptime", "sum")))

In [ ]:
lad = lad.assign(night=lad.date_obs.str[:10])
print(pd.crosstab([lad.gain, lad.exptime], lad.night, margins=True))
print()
print("achieved temperature:", sorted(lad.ccd_temp.dropna().unique()))

## Calibration matching (D9)

Which darks exist for the ladder's rungs, at gain 252 and -10 C? `FINDINGS` suspects two
gaps — 15 s darks at gain 50 only, and 240 s darks at gain 50 / -20 C. The index settles it,
and a confirmed gap is a re-shoot, never a silent substitution.

In [ ]:
dk = ok[ok.measured_type == "dark"]
print(pd.crosstab([dk.gain, dk.exptime], dk.ccd_temp.round(0), margins=True))

In [ ]:
LADDER_EXP = [15.0, 30.0, 60.0, 120.0, 240.0, 480.0]
match = dk[(dk.gain == 252) & (dk.ccd_temp.between(-10.6, -9.4))]
avail = match.groupby("exptime").size()
pd.DataFrame({"exptime": LADDER_EXP,
              "darks_g252_at_-10C": [int(avail.get(e, 0)) for e in LADDER_EXP]})